# Setup

In [1]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = "retina"

In [2]:
import os
import sys
from pprint import pprint

# so that mllm_shap can be imported without installing the package
sys.path.insert(0, os.path.abspath("../mllm_shap/src"))

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE"] = "1"
os.environ["LOG_LEVEL"] = "INFO"

In [3]:
import numpy as np
import pandas as pd
import torch

np.random.seed(42)

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"Using device: {device}")

Using device: cuda


In [4]:
from mllm_shap.connectors import TransformersCausalText, ModelConfig
from mllm_shap.connectors.enums import ModelHistoryTrackingMode, Role, SystemRolesSetup
from mllm_shap.connectors.filters import ExcludePunctuationTokensFilter, KeepAllTokens
from mllm_shap.shap import Explainer, McShapExplainer
from mllm_shap.shap.embeddings import MeanReducer
from mllm_shap.shap.enums import Mode
from mllm_shap.shap.normalizers import PowerShiftNormalizer
from mllm_shap.shap.similarity import TfIdfCosineSimilarity
from mllm_shap.utils.jupyter import display_shap_colors_df

# Usage

Define LiquidAudio model (this call loads it up to the memory!).

Create compact explainer that will make initial call and then explain it using Shapley Values approximated using Monte Carlo. Set minimal number of samples for demo purpose, that is only first-order omission ones (where only one token at the time is hidden) and empty sample (if it is not "globally" empty, that is when expandability is not performed on all tokens).

PowerNormalizer first shifts all shapley values by subtracting their minimum (so new minimal value will be 0.0), then raises them to power of 2.0 and normalizes so they sum to 1.0.

In [5]:
model = TransformersCausalText(
    device=device, history_tracking_mode=ModelHistoryTrackingMode.TEXT
)  # track and generate only text history
shap = McShapExplainer(
    num_samples=-1,
    mode=Mode.CONTEXTUAL,  # use contextual embeddings, default
    # use mean pooling to reduce token embeddings to single embedding per audio, default
    embedding_reducer=MeanReducer(),
    similarity_measure=TfIdfCosineSimilarity(),  # use TF-IDF weighted cosine similarity to compare embeddings
    normalizer=PowerShiftNormalizer(power=2.0),  # use power-shift normalization with power of 2.0
)
explainer = Explainer(model=model, shap_explainer=shap)

Create new chat that treats Assistant messages as system, that is will ignore them for shapley values calculation - they will be feed to each prompt as system messages. This significantly reduces number of requests needed for multi turn expandability, yet might not be possible due to business requirements. 

To further reduce number of calls we exclude punctuation tokens.

In [6]:
chat = model.get_new_chat(
    system_roles_setup=SystemRolesSetup.SYSTEM_ASSISTANT,
    token_filter=KeepAllTokens(),  # exclude punctuation tokens from shapley values calculation
)

chat.new_turn(Role.SYSTEM)
chat.add_text("You are a helpful assistant that answers questions briefly.")
chat.end_turn()

chat.new_turn(Role.USER)
chat.add_text("Who are you?")
chat.end_turn()

Let's have a look at chat representation:

In [7]:
chat.get_conversation()

[[ChatEntry(content_type=0, roles=[SYSTEM, SYSTEM, ..., SYSTEM, SYSTEM], content='You,  are,  a,  helpful,  assistant,  that,  answers,  questions,  briefly, ....', shap_values=None)],
 [ChatEntry(content_type=0, roles=[USER, USER, USER, USER], content='Who,  are,  you, ?', shap_values=None)]]

Representation is a list of list of ConversationEntry - so it can be accessed as  chat.get_conversation()[turn_number][message_number].{field}

Let's calculate shapley values for current conversation.

Verbose=True allows us to access history object (descried later). Generation kwargs allows to customize model interference - here we limit it to 64 tokens and change text_temperature from default 0.0 to 0.2. 

In [8]:
generation_kwargs = {"max_new_tokens": 64, "model_config": ModelConfig(text_temperature=0.2)}

result = explainer(
    chat=chat,
    verbose=True,
    generation_kwargs=generation_kwargs,
    progress_bar=True,  # show progress bar during generation, default
)

2025-11-09 15:28:16,163 - mllm_shap.shap.compact - INFO - Generating full response from the model...
/home/mvishiu11/Desktop/AudioShap_WUT_Bachelor/mllm_shap/src/mllm_shap/shap/compact.py:116: UserWarning: Audio generation parameters were provided but this connector is text-only;                     audio settings are ignored.
  response = self.model.generate(
2025-11-09 15:28:16,841 - mllm_shap.shap.base.explainer - INFO - Number of tokens for explainability: 4 (up to 15 additional calls)


Calculating SHAP values:   0%|          | 0/5 [00:00<?, ?it/s]

/home/mvishiu11/Desktop/AudioShap_WUT_Bachelor/mllm_shap/src/mllm_shap/shap/base/_generate_responses.py:132: UserWarning: Audio generation parameters were provided but this connector is text-only;                     audio settings are ignored.
  model_response = model.generate(
2025-11-09 15:28:22,595 - mllm_shap.shap.base.explainer - INFO - Deduplicated 0/5 masks using existing cache.


Result has now following fields available:

- full_chat - chat with base response (generated based on whole entry) with set cache and calculated shapley values
- source_chat - original chat feed to the explainer
- history - history of all chats used

Cache object stores actual shapley values as well as calculated embeddings and masks. They will be reused in next call regardless to the method, so for monte-carlo it is just larger sample, for precise it means some results might get excluded.

Let's first analyze history - it is a list of size equivalent to number of calls made for calculations + 1 (first entry, for base calculations, always None) - in this case, 4 (3 for base one-versus-all and one for empty call). Each entry is a tuple of following values:

- mask for that entry
- mash hash
- source chat with masked entry or None if corresponding mask was available in cache
- model response object

or None - when either corresponding mask was extracted from cache or it has risen an AllTextTokensFilteredOutError error.

Let's see all chats that were taken into account:

In [9]:
[c[2].decode_text() if c is not None else None for c in result.history]

['You are a helpful assistant that answers questions briefly.',
 'You are a helpful assistant that answers questions briefly. are you?',
 'You are a helpful assistant that answers questions briefly.Who you?',
 'You are a helpful assistant that answers questions briefly.Who are?',
 'You are a helpful assistant that answers questions briefly.Who are you']

We can see that "?" was never removed, as it is present even in the empty mask. For rest, we can see that all system tokens are always present and only user tokens get masked. out between each calls.

Let's now analyze calculated shapley values.

In [10]:
explained_chat = result.full_chat

explained_chat_conversation = explained_chat.get_conversation()
pprint(explained_chat_conversation)

[[ChatEntry(content_type=0, roles=[SYSTEM, SYSTEM, ..., SYSTEM, SYSTEM], content='You,  are,  a,  helpful,  assistant,  that,  answers,  questions,  briefly, ....', shap_values=[nan, nan, ..., nan, nan])],
 [ChatEntry(content_type=0, roles=[USER, USER, USER, USER], content='Who,  are,  you, ?', shap_values=[0.04667891561985016, 0.0, 0.9459587335586548, 0.00736238481476903])],
 [ChatEntry(content_type=0, roles=[ASSISTANT, ASSISTANT, ..., ASSISTANT, ASSISTANT], content='\n, Answer, :,  I,  am,  a,  helpful,  assistant,  that,  answers,  questions,  briefly, ., \n, <|en...', shap_values=[nan, nan, ..., nan, nan])]]


Model was set to return just text tokens, so chat history has only text tokens. We can see that in the json representation inside ConversationEntry shap_values field is now populated. Nan values indicated that this token wasn't taken into calculation scope. As expected, we have 3 not-nan tokens. Let's see them.

In [11]:
user_entry = explained_chat_conversation[1][0]

display_shap_colors_df(
    pd.DataFrame(
        list(zip(user_entry.content, user_entry.shap_values, user_entry.roles)),
        columns=["Token", "Shapley Value", "Role"],
    )
)

,Token,Shapley Value,Role
0,Who,0.046679,0
1,are,0.000000,0
2,you,0.945959,0
3,?,0.007362,0


Let's create another turn to see how input significance will change:

In [12]:
explained_chat.new_turn(Role.USER)
explained_chat.add_text("Can you repeat?")
explained_chat.end_turn()

And again, let's explain it:

In [13]:
result = explainer(chat=explained_chat, verbose=True, generation_kwargs=generation_kwargs)

2025-11-09 15:28:32,188 - mllm_shap.shap.compact - INFO - Generating full response from the model...
/home/mvishiu11/Desktop/AudioShap_WUT_Bachelor/mllm_shap/src/mllm_shap/shap/compact.py:116: UserWarning: Audio generation parameters were provided but this connector is text-only;                     audio settings are ignored.
  response = self.model.generate(
2025-11-09 15:28:33,525 - mllm_shap.shap.base.explainer - INFO - Number of tokens for explainability: 8 (up to 255 additional calls)


Calculating SHAP values:   0%|          | 0/9 [00:00<?, ?it/s]

/home/mvishiu11/Desktop/AudioShap_WUT_Bachelor/mllm_shap/src/mllm_shap/shap/base/_generate_responses.py:132: UserWarning: Audio generation parameters were provided but this connector is text-only;                     audio settings are ignored.
  model_response = model.generate(
2025-11-09 15:28:52,594 - mllm_shap.shap.base.explainer - INFO - Deduplicated 0/9 masks using existing cache.


In [14]:
[c[2].decode_text() if c is not None else None for c in result.history]

['You are a helpful assistant that answers questions briefly.\nAnswer: I am a helpful assistant that answers questions briefly.\n<|endoftext|>',
 'You are a helpful assistant that answers questions briefly. are you?\nAnswer: I am a helpful assistant that answers questions briefly.\n<|endoftext|>Can you repeat?',
 'You are a helpful assistant that answers questions briefly.Who you?\nAnswer: I am a helpful assistant that answers questions briefly.\n<|endoftext|>Can you repeat?',
 'You are a helpful assistant that answers questions briefly.Who are?\nAnswer: I am a helpful assistant that answers questions briefly.\n<|endoftext|>Can you repeat?',
 'You are a helpful assistant that answers questions briefly.Who are you\nAnswer: I am a helpful assistant that answers questions briefly.\n<|endoftext|>Can you repeat?',
 'You are a helpful assistant that answers questions briefly.Who are you?\nAnswer: I am a helpful assistant that answers questions briefly.\n<|endoftext|> you repeat?',
 'You are 

In [15]:
explained_chat = result.full_chat

explained_chat_conversation = explained_chat.get_conversation()
pprint(explained_chat_conversation)

[[ChatEntry(content_type=0, roles=[SYSTEM, SYSTEM, ..., SYSTEM, SYSTEM], content='You,  are,  a,  helpful,  assistant,  that,  answers,  questions,  briefly, ....', shap_values=[nan, nan, ..., nan, nan])],
 [ChatEntry(content_type=0, roles=[USER, USER, USER, USER], content='Who,  are,  you, ?', shap_values=[0.14414916932582855, 0.11197980493307114, 0.14414916932582855, 0.11197977513074875])],
 [ChatEntry(content_type=0, roles=[ASSISTANT, ASSISTANT, ..., ASSISTANT, ASSISTANT], content='\n, Answer, :,  I,  am,  a,  helpful,  assistant,  that,  answers,  questions,  briefly, ., \n, <|en...', shap_values=[nan, nan, ..., nan, nan])],
 [ChatEntry(content_type=0, roles=[USER, USER, USER, USER], content='Can,  you,  repeat, ?', shap_values=[0.15518003702163696, 0.0, 0.21901538968086243, 0.11354667693376541])],
 [ChatEntry(content_type=0, roles=[ASSISTANT, ASSISTANT, ..., ASSISTANT, ASSISTANT], content=' I,  said,  to,  turn,  off,  the,  lights,  before,  leaving, .,  , \n, A, :,  I,  said,  t

In [16]:
dt = []
for i in (1, 3):
    user_entry = explained_chat_conversation[i][0]
    df = pd.DataFrame(
        list(zip(user_entry.content, user_entry.shap_values, user_entry.roles)),
        columns=["Token", "Shapley Value", "Role"],
    )
    df["Turn"] = i
    dt.append(df)

df = pd.concat(dt).reset_index(drop=True)
display_shap_colors_df(df)

,Token,Shapley Value,Role,Turn
0,Who,0.144149,0,1
1,are,0.111980,0,1
2,you,0.144149,0,1
3,?,0.111980,0,1
4,Can,0.155180,0,3
5,you,0.000000,0,3
6,repeat,0.219015,0,3
7,?,0.113547,0,3
